In [20]:
import numpy as np


num_attempt_ids = 4
num_samples_per_attempt = 5
success_rate = 0.5

scores = np.random.binomial(1, success_rate, size=(num_attempt_ids, num_samples_per_attempt))

print(scores)

[[0 0 1 1 1]
 [1 1 0 0 1]
 [0 0 0 1 1]
 [1 1 0 0 0]]


## Pass@k Reward to Advantage


In [21]:
def compute_pass_at_k_advantages(scores):

    correct_per_attempt_id = scores.sum(axis=1)

    num_attempts = num_attempt_ids
    samples = num_samples_per_attempt

    rewards = np.zeros_like(scores, dtype=float)

    for i in range(num_attempts):
        for j in range(samples):
            if scores[i, j] == 1:
                rewards[i, j] = 1
            else:
                product_term = 1
                for l in range(num_attempts):
                    if l != i:
                        product_term *= (samples - correct_per_attempt_id[l]) / samples
                rewards[i, j] = 1 - product_term

    mean_rewards = rewards.mean()
    std_rewards = rewards.std()

    advantages = (rewards - mean_rewards) / std_rewards

    return advantages

advantages = compute_pass_at_k_advantages(scores)

print(advantages)
print(advantages.sum())


[[-1.44115338 -1.44115338  0.96076892  0.96076892  0.96076892]
 [ 0.96076892  0.96076892 -1.44115338 -1.44115338  0.96076892]
 [-0.64051262 -0.64051262 -0.64051262  0.96076892  0.96076892]
 [ 0.96076892  0.96076892 -0.64051262 -0.64051262 -0.64051262]]
3.26405569239796e-14


In [22]:
import random
import numpy as np

def sample_groups(num_attempts, num_samples):
    # create all possible (attempt, sample) tuples
    all_tuples = [(a, s) for a in range(num_attempts) for s in range(num_samples)]
    random.shuffle(all_tuples)

    groups = []
    while all_tuples:
        used_attempts = set()
        group = []
        remaining = []
        for attempt_id, sample_id in all_tuples:
            if attempt_id not in used_attempts:
                group.append((attempt_id, sample_id))
                used_attempts.add(attempt_id)
            else:
                remaining.append((attempt_id, sample_id))
        groups.append(group)
        all_tuples = remaining
    return groups

def compute_group_advantages(groups, scores):
    """
    For each group, compute a reward (here: 1 if any score==1 else 0),
    then standardize across groups to get group advantages.
    """
    group_rewards = np.zeros(len(groups), dtype=float)
    for i, group in enumerate(groups):
        # scores[attempt_id, sample_id] for each tuple in the group
        group_scores = np.array([scores[attempt_id, sample_id] for attempt_id, sample_id in group])
        group_rewards[i] = 1.0 if np.any(group_scores == 1) else 0.0

    mean = group_rewards.mean()
    std = group_rewards.std()
    if std == 0:
        # no variation: advantages are all zeros
        group_advantages = np.zeros_like(group_rewards)
    else:
        group_advantages = (group_rewards - mean) / std
    return group_advantages

def compute_rollout_advantages(scores, num_groupings, rng_seed=None):
    """
    For each random grouping:
      - compute group advantages,
      - add each group's advantage to every (attempt, sample) in that group.
    Finally, average per (attempt, sample) over the number of groupings.
    """
    if rng_seed is not None:
        random.seed(rng_seed)
        np.random.seed(rng_seed)

    num_attempts, num_samples = scores.shape
    rollout_advantages = np.zeros_like(scores, dtype=float)

    for _ in range(num_groupings):
        groups = sample_groups(num_attempts, num_samples)
        group_advantages = compute_group_advantages(groups, scores)

        # add each group's advantage to each member tuple in that group
        for g_idx, group in enumerate(groups):
            adv = group_advantages[g_idx]
            for attempt_id, sample_id in group:
                rollout_advantages[attempt_id, sample_id] += adv

    # average across groupings (each tuple appears exactly once per grouping)
    rollout_advantages /= float(num_groupings)
    return rollout_advantages

# Example usage:
ra = compute_rollout_advantages(scores, num_groupings=100000)
print(ra)
print(ra.sum())


[[-0.21744303 -0.20974303  0.14239536  0.14239536  0.14239536]
 [ 0.14239536  0.14239536 -0.21631803 -0.21086803  0.14239536]
 [-0.09460348 -0.09485967 -0.09532756  0.14239536  0.14239536]
 [ 0.14239536  0.14239536 -0.09775807 -0.09544385 -0.09158879]]
2.3869795029440866e-15


In [23]:
advantages/ra

array([[6.62772849, 6.87104293, 6.74719285, 6.74719285, 6.74719285],
       [6.74719285, 6.74719285, 6.66219716, 6.8343853 , 6.74719285],
       [6.77049713, 6.75221214, 6.71907092, 6.74719285, 6.74719285],
       [6.74719285, 6.74719285, 6.55201772, 6.71088444, 6.99335133]])